In [ ]:
# JUPYTER CELL 2: targeted deletion by run name pattern
# Default is DRY-RUN. Set CONFIRM=True to actually delete.

from typing import Iterable, List
from datetime import datetime, timezone
import wandb

# -------- parameters (edit these) --------
ENTITY         = "t_p-universit-t-ulm"
PROJECT        = "rl_project"   # <-- set to your project
DELETE_RUN     = True                    # delete the run after pruning artifacts/files
DELETE_ARTS    = True                    # delete logged artifacts of type 'model'
DELETE_FILES   = True                    # delete big files from runs before deletion
FILE_SUFFIXES  = (".gif", ".mp4", ".pkl")
AGENTS         = ("ddpg", "td3", "sac")
CONFIRM        =  True                # <-- flip to True to actually delete
MAX_RUNS       = None                    # int or None to process all matches
# -----------------------------------------

def human(n: int) -> str:
    for u in ["B","KB","MB","GB","TB","PB"]:
        if n < 1024:
            return f"{n:.1f}{u}"
        n /= 1024
    return f"{n:.1f}EB"

def match_run_name(name: str) -> bool:
    # Example names: test__FetchPush-v4__ddpg__shape_box
    #                test__FetchPush-v4__sac__shape_sphere
    if not name.startswith("test__FetchPush-v4__"):
        return False
    if "__fric_" not in name:
        return False
    return any(f"__{a}__" in name for a in AGENTS)

api = wandb.Api()
path = f"{ENTITY}/{PROJECT}"
runs = list(api.runs(path))  # we filter below

matched = [r for r in runs if match_run_name(r.name or "")]
if MAX_RUNS is not None:
    matched = matched[:MAX_RUNS]

print(f"Found {len(matched)} matching runs in {path} that satisfy: "
      f"'test__FetchPush-v4__{{ddpg|td3|sac}}__shape_*'")

total_art_bytes = 0
total_file_bytes = 0

for i, run in enumerate(matched, 1):
    print(f"\n[{i}/{len(matched)}] RUN {run.id} | {run.name} | state={run.state}")

    # (1) Delete logged artifacts of type 'model'
    if DELETE_ARTS:
        try:
            for art in run.logged_artifacts():
                if art.type != "model":
                    continue
                sz = getattr(art, "size", 0) or 0
                total_art_bytes += sz
                action = "DELETE" if CONFIRM else "DRY"
                print(f"  [ARTIFACT] {art.name} {art.version} size={human(sz)} -> {action}")
                if CONFIRM:
                    try:
                        art.delete()
                    except Exception as e:
                        print("    ! artifact delete failed:", e)
        except Exception as e:
            print("  ! fetching logged_artifacts failed:", e)

    # (2) Delete large files within the run
    if DELETE_FILES:
        try:
            for f in run.files():
                nm = f.name or ""
                if nm.endswith(FILE_SUFFIXES):
                    sz = f.size or 0
                    total_file_bytes += sz
                    action = "DELETE" if CONFIRM else "DRY"
                    print(f"  [RUN FILE] {nm} size={human(sz)} -> {action}")
                    if CONFIRM:
                        try:
                            f.delete()
                        except Exception as e:
                            print("    ! file delete failed:", e)
        except Exception as e:
            print("  ! listing run files failed:", e)

    # (3) Delete the run itself (optional)
    if DELETE_RUN:
        action = "DELETE" if CONFIRM else "DRY"
        print(f"  [RUN] {run.id} -> {action}")
        if CONFIRM:
            try:
                run.delete()
            except Exception as e:
                print("    ! run delete failed:", e)

print("\n----- SUMMARY -----")
print(f"Runs matched: {len(matched)}")
print(f"Artifacts processed: {human(total_art_bytes)} total")
print(f"Run files processed: {human(total_file_bytes)} total")
print("NOTE: Set CONFIRM=True and re-run to actually delete.")
